# OCR Preprocessing Pipeline

Prepare document images for Tesseract OCR extraction.
Goal: maximize text readability, unlike CNN preprocessing which preserves visual features.

Pipeline: check feasible -> enhance contrast (CLAHE) -> deskew -> binarize (Otsu) -> denoise -> resize -> save

Key differences from CNN preprocessing:
- Blank threshold same as CNN (0.997), but blanks are SKIPPED (unlike CNN which keeps them)
- Blank and dark images are SKIPPED (OCR yields nothing useful)
- No 224x224 resize: OCR needs high resolution
- Binarize, denoise, enhance contrast (not done for CNN)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict, Counter

from deskew import determine_skew

plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# paths
INPUT_DIR   = Path('rvlcdip_split')
OUTPUT_DIR  = Path('OCR_preprocessed')       # processed for OCR
OUTPUT_DIR_HR = Path('OCR_preprocessed_hr')  # high-res for report

SPLITS = ['train', 'val', 'test']

CLASSES = [
    'Advertisement',
    'Budget',
    'Email',
    'FileFolder',
    'Form',
    'Handwritten',
    'Invoice',
    'Letter',
    'Memo',
    'News',
    'Presentation',
    'Questionnaire',
    'Resume',
    'ScientificPub',
    'ScientificReport',
    'Specification',
]

NUM_CLASSES = len(CLASSES)

# OCR quality thresholds
BLANK_THRESHOLD_OCR = 0.997  # same as CNN; emails/letters are >0.95 white but have text
DARK_THRESHOLD_OCR  = 0.6    # same as CNN

# filtering decisions
SKIP_BLANKS = True    # blank pages have no text for OCR
SKIP_DARK   = True    # dark images are unreadable by OCR

# CLAHE parameters
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_SIZE  = (8, 8)

# denoising kernel
DENOISE_KERNEL = 1  # small kernel to avoid erasing thin strokes

# resize: upscale if height below this
MIN_HEIGHT = 1000

# report figure DPI
FIG_DPI = 600

print(f"Classes: {NUM_CLASSES}")
print(f"OCR blank threshold: {BLANK_THRESHOLD_OCR}")
print(f"OCR dark threshold:  {DARK_THRESHOLD_OCR}")
print(f"Skip blanks: {SKIP_BLANKS}")
print(f"Skip dark:   {SKIP_DARK}")
print(f"CLAHE clip: {CLAHE_CLIP_LIMIT}, tile: {CLAHE_TILE_SIZE}")
print(f"Denoise kernel: {DENOISE_KERNEL}")
print(f"Min height for resize: {MIN_HEIGHT}px")
print(f"Report DPI: {FIG_DPI}")

In [ ]:
# verify train/val/test splits with class folders
print("Verifying split structure...")
print(f"Input directory: {INPUT_DIR.resolve()}")
print()

all_ok = True
for split in SPLITS:
    print(f"  {split}/")
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            print(f"    NOT FOUND: {cls}")
            all_ok = False
            continue
        count = len([f for f in os.listdir(cls_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])
        print(f"    {cls:<22} {count:>5} images")
    print()

if all_ok:
    total = sum(
        len([f for f in os.listdir(INPUT_DIR / s / c)
             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])
        for s in SPLITS for c in CLASSES if (INPUT_DIR / s / c).exists()
    )
    print(f"All splits found. Total images: {total}")
else:
    print("Some folders missing. Check paths.")

In [ ]:
def is_blank(img_path, threshold=BLANK_THRESHOLD_OCR):
    """Check if an image is mostly blank (white).
    Same threshold as CNN (0.997); emails/letters exceed 0.95 but have text.
    Returns (is_blank: bool, white_ratio: float)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True, 1.0
    white_ratio = np.sum(img > 240) / img.size
    return white_ratio >= threshold, white_ratio


def is_dark(img_path, threshold=DARK_THRESHOLD_OCR):
    """Check if an image is mostly dark (black).
    Returns (is_dark: bool, dark_ratio: float)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False, 0.0
    dark_ratio = np.sum(img < 15) / img.size
    return dark_ratio >= threshold, dark_ratio


def check_ocr_feasible(img_path):
    """Combined gate: check if OCR can extract useful text.
    Returns (feasible: bool, reason: str)."""
    blank, _ = is_blank(img_path)
    if blank:
        return False, "blank"
    dark, _ = is_dark(img_path)
    if dark:
        return False, "dark"
    return True, "ok"


# test on one image per class from train
print("Testing OCR feasibility on one train sample per class:")
print("-" * 62)
for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample
    feasible, reason = check_ocr_feasible(path)
    status = "SKIP (" + reason + ")" if not feasible else "OK"
    print(f"  {cls:<22} {status}")

In [ ]:
# show blank detection with OCR threshold (0.997) for 5 samples from 4 classes
sample_classes = ['FileFolder', 'Form', 'Letter', 'Email']

fig, axes = plt.subplots(4, 5, figsize=(15, 10))
fig.suptitle("Blank Check with OCR Threshold (0.997)", fontsize=14)

for row, cls in enumerate(sample_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))[:5]
    for col, fname in enumerate(files):
        ax = axes[row][col]
        img = Image.open(cls_dir / fname)
        ax.imshow(img, cmap='gray')
        _, ratio = is_blank(cls_dir / fname)
        color = 'red' if ratio >= BLANK_THRESHOLD_OCR else 'black'
        ax.set_title(f"{ratio:.3f}", fontsize=9, color=color)
        ax.axis('off')
    axes[row][0].set_ylabel(cls, fontsize=10, rotation=0, labelpad=60)

plt.tight_layout()
plt.show()

In [ ]:
# scan all images for OCR feasibility
print("Scanning for OCR feasibility...")
feasibility = {}  # (split, cls) -> {'blank': int, 'dark': int, 'ok': int, 'total': int}

for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        counts = {'blank': 0, 'dark': 0, 'ok': 0, 'total': 0}
        if not cls_dir.exists():
            feasibility[(split, cls)] = counts
            continue
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue
            counts['total'] += 1
            feasible, reason = check_ocr_feasible(cls_dir / fname)
            if not feasible:
                counts[reason] += 1
            else:
                counts['ok'] += 1
        feasibility[(split, cls)] = counts

print(f"{'Class':<22} {'Total':>5} {'Blank':>5} {'Dark':>5} {'OK':>5} {'Skip%':>6}")
print("-" * 56)

grand_total = 0
grand_skip = 0
for cls in CLASSES:
    t = sum(feasibility.get((s, cls), {}).get('total', 0) for s in SPLITS)
    b = sum(feasibility.get((s, cls), {}).get('blank', 0) for s in SPLITS)
    d = sum(feasibility.get((s, cls), {}).get('dark', 0) for s in SPLITS)
    o = sum(feasibility.get((s, cls), {}).get('ok', 0) for s in SPLITS)
    skip_pct = (b + d) / t * 100 if t > 0 else 0
    print(f"  {cls:<20} {t:>5} {b:>5} {d:>5} {o:>5} {skip_pct:>5.1f}%")
    grand_total += t
    grand_skip += b + d

print("-" * 56)
print(f"  {'TOTAL':<18} {grand_total:>5} {grand_skip:>5} will be skipped ({grand_skip/grand_total*100:.1f}%)")

In [ ]:
# visualize skipped images (blank + dark) for report
skip_images = []  # (path, cls, reason, ratio)

for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            continue
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue
            path = cls_dir / fname
            feasible, reason = check_ocr_feasible(path)
            if not feasible:
                blank, wr = is_blank(path)
                dark, dr = is_dark(path)
                ratio = wr if reason == 'blank' else dr
                skip_images.append((path, cls, reason, ratio))

# show up to 20 skip images
n_show = min(20, len(skip_images))
if n_show > 0:
    cols = 5
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2), dpi=200)
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    fig.suptitle("Images Skipped for OCR (blank or dark)",
                 fontsize=14, fontweight='bold', y=1.01)

    for idx in range(n_show):
        path, cls, reason, ratio = skip_images[idx]
        r, c = divmod(idx, cols)
        ax = axes[r][c] if rows > 1 else axes[c]
        img = Image.open(path)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"{cls}\n{reason}={ratio:.3f}", fontsize=8, pad=4)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for idx in range(n_show, rows * cols):
        r, c = divmod(idx, cols)
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('ocr_skipped_images.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Showing {n_show} of {len(skip_images)} skipped images")
else:
    print("No images to skip.")

In [ ]:
def enhance_contrast(img_array):
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization).
    Works on grayscale or single-channel array.
    Returns enhanced image."""
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_array

    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_SIZE)
    enhanced = clahe.apply(gray)
    return enhanced


# test on 4 classes, show before/after
test_classes = ['Invoice', 'Letter', 'Memo', 'ScientificPub']

fig, axes = plt.subplots(2, len(test_classes), figsize=(4 * len(test_classes), 7), dpi=150)
fig.suptitle("CLAHE Contrast Enhancement", fontsize=14, fontweight='bold')

for col, cls in enumerate(test_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    # pick a non-blank, non-dark sample
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break

    img = cv2.imread(str(path))
    enhanced = enhance_contrast(img)

    axes[0][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(cls, fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(enhanced, cmap='gray')
    axes[1][col].set_title("After CLAHE", fontsize=9)
    axes[1][col].axis('off')

axes[0][0].set_ylabel("Original", fontsize=11)
axes[1][0].set_ylabel("CLAHE", fontsize=11)
plt.tight_layout()
plt.savefig('clahe_before_after.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# show histogram before/after CLAHE for one sample
sample_cls = 'Invoice'
cls_dir = INPUT_DIR / 'train' / sample_cls
files = sorted(os.listdir(cls_dir))
for fname in files:
    path = cls_dir / fname
    feasible, _ = check_ocr_feasible(path)
    if feasible:
        break

img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
enhanced = enhance_contrast(img)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
fig.suptitle(f"Histogram: {sample_cls} sample", fontsize=13, fontweight='bold')

axes[0].hist(img.ravel(), bins=256, range=(0, 256), color='steelblue', alpha=0.8)
axes[0].set_title("Before CLAHE", fontsize=10)
axes[0].set_xlabel("Pixel Value")
axes[0].set_ylabel("Count")

axes[1].hist(enhanced.ravel(), bins=256, range=(0, 256), color='darkorange', alpha=0.8)
axes[1].set_title("After CLAHE", fontsize=10)
axes[1].set_xlabel("Pixel Value")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig('clahe_histogram.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def fix_skew(img_array):
    """Rotate image to correct skew using deskew library.
    Works on grayscale input. Returns (corrected_array, angle)."""
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_array
    angle = determine_skew(gray)
    if angle is None or abs(angle) < 0.5:
        return img_array, 0.0
    h, w = img_array.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(img_array, matrix, (w, h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


# test deskew AFTER CLAHE on 4 classes
test_classes = ['Invoice', 'Letter', 'Form', 'Budget']

fig, axes = plt.subplots(3, len(test_classes), figsize=(4 * len(test_classes), 10), dpi=150)
fig.suptitle("CLAHE then Deskew Pipeline", fontsize=14, fontweight='bold')

for col, cls in enumerate(test_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break

    img = cv2.imread(str(path))
    enhanced = enhance_contrast(img)
    corrected, angle = fix_skew(enhanced)

    axes[0][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(cls, fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(enhanced, cmap='gray')
    axes[1][col].set_title("After CLAHE", fontsize=9)
    axes[1][col].axis('off')

    axes[2][col].imshow(corrected, cmap='gray')
    axes[2][col].set_title(f"Deskew {angle:+.1f} deg", fontsize=9)
    axes[2][col].axis('off')

axes[0][0].set_ylabel("Original", fontsize=11)
axes[1][0].set_ylabel("CLAHE", fontsize=11)
axes[2][0].set_ylabel("Deskewed", fontsize=11)
plt.tight_layout()
plt.savefig('clahe_deskew_pipeline.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def binarize(img_array):
    """Binarize grayscale image using Otsu's threshold.
    Returns binary image (0 or 255)."""
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_array
    # Otsu automatically finds the optimal threshold
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


# test binarize after CLAHE + deskew on 4 classes
test_classes = ['Invoice', 'Letter', 'Form', 'Budget']

fig, axes = plt.subplots(4, len(test_classes), figsize=(4 * len(test_classes), 13), dpi=150)
fig.suptitle("Full Pipeline: CLAHE -> Deskew -> Binarize", fontsize=14, fontweight='bold')

for col, cls in enumerate(test_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break

    img = cv2.imread(str(path))
    enhanced = enhance_contrast(img)
    deskewed, angle = fix_skew(enhanced)
    binary = binarize(deskewed)

    axes[0][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(cls, fontsize=10)
    axes[0][col].axis('off')

    axes[1][col].imshow(enhanced, cmap='gray')
    axes[1][col].set_title("CLAHE", fontsize=9)
    axes[1][col].axis('off')

    axes[2][col].imshow(deskewed, cmap='gray')
    axes[2][col].set_title(f"Deskew {angle:+.1f} deg", fontsize=9)
    axes[2][col].axis('off')

    axes[3][col].imshow(binary, cmap='gray')
    axes[3][col].set_title("Otsu Binary", fontsize=9)
    axes[3][col].axis('off')

axes[0][0].set_ylabel("Original", fontsize=11)
axes[1][0].set_ylabel("CLAHE", fontsize=11)
axes[2][0].set_ylabel("Deskewed", fontsize=11)
axes[3][0].set_ylabel("Binarized", fontsize=11)
plt.tight_layout()
plt.savefig('clahe_deskew_binarize.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def remove_noise(img_array, kernel_size=DENOISE_KERNEL):
    """Remove small noise using morphological opening.
    Opening = erosion followed by dilation.
    Removes small dots/speckles without destroying text strokes.
    Expects binary (0/255) image."""
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    cleaned = cv2.morphologyEx(img_array, cv2.MORPH_OPEN, kernel)
    return cleaned


# test denoising after binarize on 4 classes
test_classes = ['Invoice', 'Letter', 'Handwritten', 'News']

fig, axes = plt.subplots(2, len(test_classes), figsize=(4 * len(test_classes), 7), dpi=150)
fig.suptitle("Binarize -> Denoise", fontsize=14, fontweight='bold')

for col, cls in enumerate(test_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break

    img = cv2.imread(str(path))
    enhanced = enhance_contrast(img)
    deskewed, _ = fix_skew(enhanced)
    binary = binarize(deskewed)
    denoised = remove_noise(binary)

    axes[0][col].imshow(binary, cmap='gray')
    axes[0][col].set_title(f"{cls} (binary)", fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(denoised, cmap='gray')
    axes[1][col].set_title("Denoised", fontsize=9)
    axes[1][col].axis('off')

axes[0][0].set_ylabel("Binary", fontsize=11)
axes[1][0].set_ylabel("Denoised", fontsize=11)
plt.tight_layout()
plt.savefig('binarize_denoise.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def resize_for_ocr(img_array, min_height=MIN_HEIGHT):
    """Upscale image if height is below min_height.
    Tesseract works best at 300 DPI (~1000px height).
    If already large enough, return as-is.
    Returns resized image."""
    h, w = img_array.shape[:2]
    if h < min_height:
        scale = min_height / h
        new_w = int(w * scale)
        resized = cv2.resize(img_array, (new_w, min_height), interpolation=cv2.INTER_CUBIC)
        return resized
    return img_array


# test resize on images with different heights
print("Testing resize_for_ocr:")
print("-" * 50)
for cls in CLASSES[:8]:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape[:2]
    resized = resize_for_ocr(img)
    rh, rw = resized.shape[:2]
    status = f"upscaled to {rh}x{rw}" if rh != h else f"kept {rh}x{rw}"
    print(f"  {cls:<22} {h}x{w} -> {status}")

In [ ]:
# visualize resize effect on a small image
# find the smallest feasible image across train
min_h = 99999
min_path = None
min_cls = None

for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if not feasible:
            continue
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is not None and img.shape[0] < min_h:
            min_h = img.shape[0]
            min_path = path
            min_cls = cls

if min_path is not None:
    img = cv2.imread(str(min_path), cv2.IMREAD_GRAYSCALE)
    resized = resize_for_ocr(img)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=150)
    fig.suptitle(f"Resize for OCR ({min_cls}, smallest train image)", fontsize=13, fontweight='bold')

    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f"Original {img.shape[1]}x{img.shape[0]}", fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(resized, cmap='gray')
    axes[1].set_title(f"Resized {resized.shape[1]}x{resized.shape[0]}", fontsize=10)
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig('resize_for_ocr.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("No feasible images found for resize test.")

In [ ]:
def preprocess_ocr(img_path, output_path):
    """Full OCR preprocessing pipeline.
    check feasible -> CLAHE -> deskew -> binarize -> denoise -> resize -> save
    Returns (success: bool, reason: str).
    reason is 'ok' on success, or 'blank'/'dark'/'fail' on skip/error."""
    # check feasibility first
    feasible, reason = check_ocr_feasible(img_path)
    if not feasible:
        return False, reason

    # load image
    img = cv2.imread(str(img_path))
    if img is None:
        return False, 'fail'

    # pipeline: CLAHE -> deskew -> binarize -> denoise -> resize
    enhanced = enhance_contrast(img)
    deskewed, _ = fix_skew(enhanced)
    binary = binarize(deskewed)
    denoised = remove_noise(binary)
    final = resize_for_ocr(denoised)

    # save as PNG (lossless, JPEG artifacts hurt OCR)
    out_dir = os.path.dirname(str(output_path))
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    cv2.imwrite(str(output_path), final, [cv2.IMWRITE_PNG_COMPRESSION, 0])
    return True, 'ok'


def preprocess_ocr_hr(img_path, output_path):
    """High-res version for report: same pipeline but no resize.
    Returns (success: bool, reason: str)."""
    feasible, reason = check_ocr_feasible(img_path)
    if not feasible:
        return False, reason

    img = cv2.imread(str(img_path))
    if img is None:
        return False, 'fail'

    enhanced = enhance_contrast(img)
    deskewed, _ = fix_skew(enhanced)
    binary = binarize(deskewed)
    denoised = remove_noise(binary)
    # no resize: keep original dimensions

    out_dir = os.path.dirname(str(output_path))
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    cv2.imwrite(str(output_path), denoised, [cv2.IMWRITE_PNG_COMPRESSION, 0])
    return True, 'ok'


# test both functions on one sample
test_cls = 'Invoice'
test_dir = INPUT_DIR / 'train' / test_cls
files = sorted(os.listdir(test_dir))
for fname in files:
    path = test_dir / fname
    feasible, _ = check_ocr_feasible(path)
    if feasible:
        break

test_in = test_dir / fname

# test standard pipeline
out_std = Path("test_ocr_output.png")
success, reason = preprocess_ocr(test_in, out_std)
if success:
    img_out = Image.open(out_std)
    print(f"Standard: {img_out.size} OK")
    out_std.unlink()
else:
    print(f"Standard: SKIPPED ({reason})")

# test high-res pipeline
out_hr = Path("test_ocr_hr_output.png")
success_hr, reason_hr = preprocess_ocr_hr(test_in, out_hr)
if success_hr:
    img_hr = Image.open(out_hr)
    print(f"High-res: {img_hr.size} OK")
    out_hr.unlink()
else:
    print(f"High-res: SKIPPED ({reason_hr})")

In [ ]:
# show full pipeline step by step for 4 classes
test_classes = ['Invoice', 'Letter', 'ScientificPub', 'Resume']

fig, axes = plt.subplots(6, len(test_classes), figsize=(4 * len(test_classes), 18), dpi=150)
fig.suptitle("OCR Preprocessing: Step by Step", fontsize=14, fontweight='bold')

step_labels = ["Original", "CLAHE", "Deskewed", "Binarized", "Denoised", "Resized"]

for col, cls in enumerate(test_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        path = cls_dir / fname
        feasible, _ = check_ocr_feasible(path)
        if feasible:
            break

    img = cv2.imread(str(path))
    enhanced = enhance_contrast(img)
    deskewed, angle = fix_skew(enhanced)
    binary = binarize(deskewed)
    denoised = remove_noise(binary)
    final = resize_for_ocr(denoised)

    steps = [img, enhanced, deskewed, binary, denoised, final]
    for row, step_img in enumerate(steps):
        if len(step_img.shape) == 3:
            axes[row][col].imshow(cv2.cvtColor(step_img, cv2.COLOR_BGR2RGB))
        else:
            axes[row][col].imshow(step_img, cmap='gray')
        if row == 0:
            axes[row][col].set_title(cls, fontsize=10)
        axes[row][col].axis('off')

for row, label in enumerate(step_labels):
    axes[row][0].set_ylabel(label, fontsize=10, rotation=0, labelpad=60)

plt.tight_layout()
plt.savefig('ocr_pipeline_steps.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# process all images: save to OCR_preprocessed/ and OCR_preprocessed_hr/
total_ok = 0
total_blank = 0
total_dark = 0
total_fail = 0

for split in SPLITS:
    print(f"Processing {split}...")
    split_ok = 0
    split_blank = 0
    split_dark = 0
    split_fail = 0

    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            continue
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue

            in_path = cls_dir / fname

            # save processed version for OCR
            out_path = OUTPUT_DIR / split / cls / (Path(fname).stem + '.png')
            success, reason = preprocess_ocr(in_path, out_path)

            # save high-res version for report
            if success:
                out_hr_path = OUTPUT_DIR_HR / split / cls / (Path(fname).stem + '.png')
                preprocess_ocr_hr(in_path, out_hr_path)

            if success:
                split_ok += 1
            elif reason == 'blank':
                split_blank += 1
            elif reason == 'dark':
                split_dark += 1
            else:
                split_fail += 1

    print(f"  {split}: {split_ok} ok, {split_blank} blank, {split_dark} dark, {split_fail} failed")
    total_ok += split_ok
    total_blank += split_blank
    total_dark += split_dark
    total_fail += split_fail

print()
print(f"Total: {total_ok} processed, {total_blank} blank skipped, {total_dark} dark skipped, {total_fail} failed")
print(f"OCR output:  {OUTPUT_DIR}/ (processed, resized)")
print(f"Report:      {OUTPUT_DIR_HR}/ (high-res, no resize)")

In [ ]:
# verify output structure and counts
print("Output directory:", OUTPUT_DIR.resolve())
print()
print(f"{'Class':<22} {'Train':>5} {'Val':>6} {'Test':>6} {'Total':>6} {'Skipped':>7}")
print("-" * 56)

totals = {'train': 0, 'val': 0, 'test': 0}
total_skipped = 0
for cls in CLASSES:
    counts = {}
    for split in SPLITS:
        d = OUTPUT_DIR / split / cls
        n = len(os.listdir(d)) if d.exists() else 0
        counts[split] = n
        totals[split] += n

    # count skipped for this class
    skipped = sum(
        feasibility.get((s, cls), {}).get('blank', 0) + feasibility.get((s, cls), {}).get('dark', 0)
        for s in SPLITS
    )
    total_skipped += skipped
    total = sum(counts.values())
    print(f"  {cls:<20} {counts['train']:>5} {counts['val']:>6} {counts['test']:>6} {total:>6} {skipped:>7}")

print("-" * 56)
grand = sum(totals.values())
print(f"  {'TOTAL':<18} {totals['train']:>5} {totals['val']:>6} {totals['test']:>6} {grand:>6} {total_skipped:>7}")

In [ ]:
# show one preprocessed OCR sample per class from output
fig, axes = plt.subplots(2, 8, figsize=(20, 5), dpi=150)
fig.suptitle("Final OCR Output Samples (train, one per class)", fontsize=14)

shown = 0
for idx, cls in enumerate(CLASSES):
    row, col = divmod(idx, 8)
    ax = axes[row][col]
    cls_dir = OUTPUT_DIR / 'train' / cls
    if not cls_dir.exists() or not os.listdir(cls_dir):
        ax.axis('off')
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    img = Image.open(cls_dir / sample)
    ax.imshow(img, cmap='gray')
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
    shown += 1

plt.tight_layout()
plt.savefig('ocr_final_samples.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Shown {shown} classes with OCR output")

In [ ]:
# comparison: CNN vs OCR preprocessing side by side for 4 classes
CNN_DIR = Path('CNN_preprocessed')
test_classes = ['Invoice', 'Letter', 'Form', 'Resume']

fig, axes = plt.subplots(2, len(test_classes), figsize=(4 * len(test_classes), 7), dpi=150)
fig.suptitle("CNN vs OCR Preprocessing Comparison", fontsize=14, fontweight='bold')

for col, cls in enumerate(test_classes):
    # CNN version
    cnn_dir = CNN_DIR / 'train' / cls
    if cnn_dir.exists() and os.listdir(cnn_dir):
        cnn_sample = sorted(os.listdir(cnn_dir))[0]
        cnn_img = Image.open(cnn_dir / cnn_sample)
        axes[0][col].imshow(cnn_img)
    else:
        axes[0][col].text(0.5, 0.5, 'N/A', ha='center', va='center')
    axes[0][col].set_title(cls, fontsize=10)
    axes[0][col].axis('off')

    # OCR version
    ocr_dir = OUTPUT_DIR / 'train' / cls
    if ocr_dir.exists() and os.listdir(ocr_dir):
        ocr_sample = sorted(os.listdir(ocr_dir))[0]
        ocr_img = Image.open(ocr_dir / ocr_sample)
        axes[1][col].imshow(ocr_img, cmap='gray')
    else:
        axes[1][col].text(0.5, 0.5, 'Skipped', ha='center', va='center')
    axes[1][col].axis('off')

axes[0][0].set_ylabel("CNN\n(224x224 color)", fontsize=10, rotation=0, labelpad=60)
axes[1][0].set_ylabel("OCR\n(binary, resized)", fontsize=10, rotation=0, labelpad=60)
plt.tight_layout()
plt.savefig('cnn_vs_ocr_comparison.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# summary: per-class OCR output count vs original count
print("Per-class OCR output vs original:")
print(f"{'Class':<22} {'Original':>8} {'OCR Out':>8} {'Skipped':>8} {'Keep%':>7}")
print("-" * 58)

for cls in CLASSES:
    # original count
    orig = 0
    for split in SPLITS:
        cls_dir = INPUT_DIR / split / cls
        if cls_dir.exists():
            orig += len([f for f in os.listdir(cls_dir)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])

    # OCR output count
    ocr_out = 0
    for split in SPLITS:
        ocr_dir = OUTPUT_DIR / split / cls
        if ocr_dir.exists():
            ocr_out += len(os.listdir(ocr_dir))

    skipped = orig - ocr_out
    keep_pct = ocr_out / orig * 100 if orig > 0 else 0
    print(f"  {cls:<20} {orig:>8} {ocr_out:>8} {skipped:>8} {keep_pct:>6.1f}%")

print("-" * 58)